# Stratification timing report

This notebook reads already-computed MacStudio timing CSVs and plots benchmark results.
It does not run benchmarks again.

Data sources, in priority order:
- `timing/stratify-fast-quickder-cpu-gpu-60s-from300-unified.csv`
- `timing/stratify-fast-stable-cpu-gpu-30s.csv`
- `timing/stratify-slow-core-cpu-gpu-60s.csv`
- `timing/stratify-timing-MacStudio.csv`
- any batch shard files matching `timing/stratify-b*.csv`

The report derives a `profile` label from `(eltype, solve_tol)` and shows:
- median + min/max error bars by dimension and method
- log-scale runtime and residual views per profile
- dedicated high-d (d >= 125) CPU vs GPU comparisons

In [ ]:
using CSV, DataFrames, PlotlyJS, Statistics, Printf
PlotlyJS.templates.default = "plotly_white"

const COLTYPES = Dict(
    :d => Int,
    :valence => Int,
    :eltype => String,
    :solve_tol => Float64,
    :target => Float64,
    :ops => String,
    :method => String,
    :solver => String,
    :threads => Int,
    :heap_hint_gb => Float64,
    :timeout_s => Float64,
    :der_seconds => Float64,
    :strat_seconds => Float64,
    :total_seconds => Float64,
    :bytes => Int,
    :nullity => Int,
    :residual => Float64,
    :meets_target => Bool,
    :lsq_err => Float64,
    :support => Float64,
    :perm_ok => Bool,
    :dims => String,
    :nnz => Int,
    :status => String,
)

function read_clean_csv(path::String)
    lines = readlines(path)
    isempty(lines) && error("$path is empty")
    want = count(==(','), lines[1])
    keep = [lines[1]; filter(l -> count(==(','), l) == want, lines[2:end])]
    CSV.read(IOBuffer(join(keep, "\n")), DataFrame; types = COLTYPES)
end

function load_timing()
    base = @__DIR__
    preferred = [
        "stratify-fast-quickder-cpu-gpu-60s-from300-unified.csv",
        "stratify-fast-stable-cpu-gpu-30s.csv",
        "stratify-slow-core-cpu-gpu-60s.csv",
        "stratify-fast-cpu-gpu-30s.csv",
        "stratify-slow-cpu-gpu-60s.csv",
        "stratify-timing-MacStudio.csv",
    ]
    shard_files = sort(filter(n -> occursin(r"^stratify-b\\d+\\.csv$", n), readdir(base)))
    shard_paths = [joinpath(base, f) for f in shard_files if isfile(joinpath(base, f))]

    paths = String[]
    for f in preferred
        p = joinpath(base, f)
        isfile(p) && push!(paths, p)
    end
    append!(paths, shard_paths)
    isempty(paths) && error("No timing CSV found in $base")

    dfs = [read_clean_csv(p) for p in paths]
    df = length(dfs) == 1 ? dfs[1] : vcat(dfs...; cols = :setequal)

    df.profile = [string(r.eltype, " (tol=", @sprintf("%.0e", r.solve_tol), ")") for r in eachrow(df)]
    return df, paths
end

df, data_paths = load_timing()
ok = filter(:status => ==("ok"), df)
ok = filter([:total_seconds, :der_seconds, :strat_seconds, :residual] =>
    (t, d, s, r) -> isfinite(t) && t > 0 && isfinite(d) && d >= 0 && isfinite(s) && s >= 0 && isfinite(r) && r >= 0,
    ok)

println("loaded files:")
foreach(p -> println("  - ", p), data_paths)
println("rows = ", nrow(df))
println("successful valid rows = ", nrow(ok))
if nrow(ok) > 0
    println("d range = ", minimum(ok.d), " to ", maximum(ok.d))
end

In [ ]:
profile_meta = combine(groupby(df, [:profile, :threads, :heap_hint_gb, :timeout_s]),
    nrow => :rows)
profile_meta = sort(profile_meta, :profile)

profile_summary = combine(groupby(df, [:profile, :method]), nrow => :trials,
    :d => minimum => :d_min,
    :d => maximum => :d_max,
    :total_seconds => median => :median_time,
    :residual => median => :median_residual)

profile_meta, sort(profile_summary, [:profile, :median_time])

In [ ]:
method_order = [
    "Auto",
    "QuickDer/Auto", "QuickDer/Arpack", "QuickDer/Gram", "QuickDer3",
    "SylverLining/Auto", "SylverLining/Gram", "SylverLining/LU",
    "SylverLining/Arpack", "SylverLining/CG", "SylverLining/Krylov",
    "SylverLining/Lanczos", "SylverLining/LSMR", "SylverLining/ShiftInvert",
    "SylverLining/SVD", "SymmetricGram",
    "CPU/Auto", "CPU/QuickDer/Auto", "CPU/QuickDer/Arpack", "CPU/QuickDer/Gram", "CPU/QuickDer3",
    "CPU/SylverLining/Auto", "CPU/SylverLining/Gram", "CPU/SylverLining/LU",
    "CPU/SylverLining/Arpack", "CPU/SylverLining/Krylov", "CPU/SylverLining/Lanczos",
    "CPU/SylverLining/ShiftInvert", "CPU/SylverLining/SVD", "CPU/SymmetricGram",
    "GPU/QuickDer/Gram", "GPU/SylverLining/Auto", "GPU/SylverLining/Gram",
]

# Keep runtime rendering light by default.
const SHOW_ERROR_BARS = false

# IMPORTANT: aggregate from successful/valid rows only.
profile_stats = combine(groupby(ok, [:profile, :method, :d]),
    :total_seconds => median => :median_t,
    :total_seconds => minimum => :min_t,
    :total_seconds => maximum => :max_t,
    :residual => median => :median_residual,
    :residual => minimum => :min_residual,
    :residual => maximum => :max_residual)

sort!(profile_stats, [:profile, :method, :d])

method_rank = Dict(m => i for (i, m) in enumerate(method_order))

profile_stats

In [ ]:
# Runtime + memory table by method and dimension (side-by-side).
function time_memory_table(data; methods = nothing, dmin::Int = 0)
    base = filter([:d, :total_seconds, :bytes] =>
        (d, t, b) -> d >= dmin && isfinite(t) && t > 0 && isfinite(b) && b > 0,
        data,
    )
    methods !== nothing && (base = filter(:method => m -> m in methods, base))
    isempty(base) && error("No rows available for time/memory table")

    grp = combine(groupby(base, [:profile, :method, :d]),
        :total_seconds => median => :median_total_seconds,
        :bytes => median => :median_bytes,
        nrow => :n,
    )
    grp.median_gib = grp.median_bytes ./ (1024.0^3)

    grp.method_rank = [get(method_rank, m, typemax(Int)) for m in grp.method]
    sort!(grp, [:profile, :d, :method_rank])

    select!(grp, [:profile, :method, :d, :median_total_seconds, :median_bytes, :median_gib, :n])
    return grp
end

focus_methods = Set(["CPU/Auto", "CPU/QuickDer/Arpack", "CPU/QuickDer/Gram", "GPU/QuickDer/Gram"])
time_mem = time_memory_table(ok; methods = focus_methods, dmin = 125)
time_mem

In [ ]:
# Plot the side-by-side time/memory table.
function time_memory_figure(tbl)
    sub = filter(:profile => ==("Float32 (tol=1e-08)"), tbl)
    isempty(sub) && error("No Float32 rows available in time_mem table")

    p = PlotlyJS.Plot()
    methods = unique(sub.method)
    for m in methods
        sm = sort(filter(:method => ==(m), sub), :d)
        add_trace!(p, scatter(
            x = sm.d,
            y = sm.median_total_seconds,
            name = string(m, " time"),
            mode = "lines+markers",
            yaxis = "y",
        ))
        add_trace!(p, scatter(
            x = sm.d,
            y = sm.median_gib,
            name = string(m, " memory"),
            mode = "lines+markers",
            line = attr(dash = "dot"),
            yaxis = "y2",
        ))
    end

    relayout!(p;
        title_text = "Time and Memory by Method and Dimension (Float32, universal)",
        width = 1250,
        height = 540,
        xaxis = attr(title = "d"),
        yaxis = attr(title = "seconds", type = "log"),
        yaxis2 = attr(title = "GiB", overlaying = "y", side = "right"),
        legend = attr(orientation = "h", y = -0.22, x = 0.0),
        margin = attr(l = 70, r = 80, t = 60, b = 120),
    )

    return p
end

tm_fig = time_memory_figure(time_mem)
tm_html = joinpath(@__DIR__, "stratify-time-memory-report.html")
savefig(tm_fig, tm_html)
println("wrote ", tm_html)
tm_fig

In [ ]:
function _plain_tick_label(v::Float64)
    if v < 1e-6
        return @sprintf("%.0e", v)
    end
    s = @sprintf("%.10f", v)
    s = replace(s, r"0+$" => "")
    s = replace(s, r"\.$" => "")
    s
end

function _runtime_log_ticks(ymin::Float64, ymax::Float64)
    lo = floor(Int, log10(ymin))
    hi = ceil(Int, log10(ymax))
    vals = Float64[]
    txt = String[]
    for e in lo:hi
        for m in (1.0, 2.0, 5.0)
            v = m * 10.0^e
            if ymin <= v <= ymax
                push!(vals, v)
                push!(txt, _plain_tick_label(v))
            end
        end
    end
    vals, txt
end

function runtime_figure(data)
    plot_data = data[isfinite.(data.median_t) .& (data.median_t .> 0), :]
    isempty(plot_data) && error("No valid runtime rows to plot")

    plot_data.method_rank = [get(method_rank, m, typemax(Int)) for m in plot_data.method]
    sort!(plot_data, [:profile, :method_rank, :d])

    p = plot(
        plot_data,
        x = :d,
        y = :median_t,
        color = :method,
        facet_col = :profile,
        kind = "scatter",
        mode = "lines+markers",
    )

    if SHOW_ERROR_BARS
        for tr in p.plot.data
            m = String(get(tr.fields, :name, ""))
            sub = filter(:method => ==(m), plot_data)
            if !isempty(sub)
                tr["error_y"] = attr(
                    type = "data",
                    symmetric = false,
                    array = sub.max_t .- sub.median_t,
                    arrayminus = sub.median_t .- sub.min_t,
                    visible = true,
                    thickness = 1.2,
                    width = 2,
                )
            end
        end
    end

    ymin = minimum(plot_data.median_t)
    ymax = maximum(plot_data.median_t)
    tickvals, ticktext = _runtime_log_ticks(ymin, ymax)

    nprof = length(unique(plot_data.profile))
    for k in 1:nprof
        xa = Symbol("xaxis", k == 1 ? "" : string(k))
        ya = Symbol("yaxis", k == 1 ? "" : string(k))
        p.plot.layout[xa] = attr(title = "d")
        p.plot.layout[ya] = attr(
            title = "seconds",
            type = "log",
            tickmode = "array",
            tickvals = tickvals,
            ticktext = ticktext,
        )
    end

    relayout!(p;
        title_text = "Median total runtime by method and dimension (MacStudio data)",
        width = 1200,
        height = 500,
        legend = attr(orientation = "h", y = -0.18, x = 0.0),
        margin = attr(l = 70, r = 20, t = 60, b = 90),
    )

    p.plot
end

In [ ]:
t_build = @elapsed runtime_fig = runtime_figure(profile_stats)
println("runtime figure build_seconds = ", round(t_build, digits = 3),
        ", traces = ", length(runtime_fig.data))

runtime_html = joinpath(@__DIR__, "stratify-runtime-report.html")
savefig(runtime_fig, runtime_html)
println("wrote ", runtime_html)

runtime_fig

## High-dimension views (d >= 125)

These plots focus on frontier runs and keep CPU/GPU labels separate.

In [ ]:
function highd_runtime_figure(data; dmin::Int = 125)
    sub = filter([:d, :median_t] => (d, t) -> d >= dmin && isfinite(t) && t > 0, data)
    isempty(sub) && error("No valid high-d runtime rows with d >= $(dmin)")

    sub.method_rank = [get(method_rank, m, typemax(Int)) for m in sub.method]
    sort!(sub, [:profile, :method_rank, :d])

    p = plot(
        sub,
        x = :d,
        y = :median_t,
        color = :method,
        facet_col = :profile,
        kind = "scatter",
        mode = "lines+markers",
    )

    ymin = minimum(sub.median_t)
    ymax = maximum(sub.median_t)
    tickvals, ticktext = _runtime_log_ticks(ymin, ymax)

    nprof = length(unique(sub.profile))
    for k in 1:nprof
        xa = Symbol("xaxis", k == 1 ? "" : string(k))
        ya = Symbol("yaxis", k == 1 ? "" : string(k))
        p.plot.layout[xa] = attr(title = "d")
        p.plot.layout[ya] = attr(
            title = "seconds",
            type = "log",
            tickmode = "array",
            tickvals = tickvals,
            ticktext = ticktext,
        )
    end

    relayout!(p;
        title_text = "High-d runtime (d >= $(dmin)) by method (CPU/GPU separated)",
        width = 1300,
        height = 520,
        legend = attr(orientation = "h", y = -0.22, x = 0.0),
        margin = attr(l = 70, r = 20, t = 60, b = 110),
    )

    return p.plot
end

function highd_quickder_cpu_gpu_figure(data; dmin::Int = 125)
    methods = Set(["CPU/QuickDer/Gram", "GPU/QuickDer/Gram"])
    sub = filter([:d, :method, :median_t] =>
        (d, m, t) -> d >= dmin && (m in methods) && isfinite(t) && t > 0,
        data)
    isempty(sub) && error("No high-d CPU/GPU QuickDer Gram rows with d >= $(dmin)")

    sort!(sub, [:profile, :method, :d])
    p = plot(
        sub,
        x = :d,
        y = :median_t,
        color = :method,
        facet_col = :profile,
        kind = "scatter",
        mode = "lines+markers",
    )

    ymin = minimum(sub.median_t)
    ymax = maximum(sub.median_t)
    tickvals, ticktext = _runtime_log_ticks(ymin, ymax)

    nprof = length(unique(sub.profile))
    for k in 1:nprof
        xa = Symbol("xaxis", k == 1 ? "" : string(k))
        ya = Symbol("yaxis", k == 1 ? "" : string(k))
        p.plot.layout[xa] = attr(title = "d")
        p.plot.layout[ya] = attr(
            title = "seconds",
            type = "log",
            tickmode = "array",
            tickvals = tickvals,
            ticktext = ticktext,
        )
    end

    relayout!(p;
        title_text = "High-d CPU vs GPU: QuickDer/Gram (d >= $(dmin))",
        width = 1200,
        height = 500,
        legend = attr(orientation = "h", y = -0.22, x = 0.0),
        margin = attr(l = 70, r = 20, t = 60, b = 110),
    )

    return p.plot
end

function _highd_complexity_rows(data; dmin::Int = 125, methods::Union{Nothing, Set{String}} = nothing)
    base = filter([:d, :median_t] => (d, t) -> d >= dmin && isfinite(t) && t > 0, data)
    methods !== nothing && (base = filter(:method => m -> m in methods, base))
    isempty(base) && error("No valid high-d runtime rows for complexity with d >= $(dmin)")

    rows = DataFrame(profile=String[], method=String[], d=Int[], slope=Float64[], ma_slope=Float64[])

    for g in groupby(base, [:profile, :method])
        sub = sort(g, :d)
        ds = Int.(collect(sub.d))
        ts = Float64.(collect(sub.median_t))

        valid = isfinite.(ts) .& (ts .> 0) .& (ds .> 1)
        if count(valid) < 2
            continue
        end

        dsv = ds[valid]
        tsv = ts[valid]
        local_slope = diff(log10.(tsv)) ./ diff(log10.(dsv))
        d_slope = dsv[2:end]

        tail = 5
        for i in eachindex(local_slope)
            lo = max(1, i - tail + 1)
            avg = mean(local_slope[lo:i])
            push!(rows, (String(sub.profile[1]), String(sub.method[1]), d_slope[i], local_slope[i], avg))
        end
    end

    isempty(rows) && error("Not enough high-d points per (profile, method) to estimate complexity")

    rows.method_rank = [get(method_rank, m, typemax(Int)) for m in rows.method]
    sort!(rows, [:profile, :method_rank, :d])
    rows
end

function highd_complexity_figure(data; dmin::Int = 125)
    rows = _highd_complexity_rows(data; dmin = dmin)

    p = plot(
        rows,
        x = :d,
        y = :ma_slope,
        color = :method,
        facet_col = :profile,
        kind = "scatter",
        mode = "lines+markers",
    )

    nprof = length(unique(rows.profile))
    for k in 1:nprof
        xa = Symbol("xaxis", k == 1 ? "" : string(k))
        ya = Symbol("yaxis", k == 1 ? "" : string(k))
        p.plot.layout[xa] = attr(title = "d")
        p.plot.layout[ya] = attr(
            title = "moving avg local exponent e (tail 5)",
            showgrid = true,
        )
    end

    relayout!(p;
        title_text = "High-d smoothed local complexity exponent (d >= $(dmin))",
        width = 1300,
        height = 520,
        legend = attr(orientation = "h", y = -0.22, x = 0.0),
        margin = attr(l = 70, r = 20, t = 60, b = 110),
    )

    return p.plot
end

function highd_quickder_complexity_cpu_gpu_figure(data; dmin::Int = 125)
    methods = Set(["CPU/QuickDer/Gram", "GPU/QuickDer/Gram"])
    rows = _highd_complexity_rows(data; dmin = dmin, methods = methods)

    p = plot(
        rows,
        x = :d,
        y = :ma_slope,
        color = :method,
        facet_col = :profile,
        kind = "scatter",
        mode = "lines+markers",
    )

    nprof = length(unique(rows.profile))
    for k in 1:nprof
        xa = Symbol("xaxis", k == 1 ? "" : string(k))
        ya = Symbol("yaxis", k == 1 ? "" : string(k))
        p.plot.layout[xa] = attr(title = "d")
        p.plot.layout[ya] = attr(
            title = "moving avg local exponent e (tail 5)",
            showgrid = true,
        )
    end

    relayout!(p;
        title_text = "High-d CPU vs GPU complexity: QuickDer/Gram (d >= $(dmin))",
        width = 1200,
        height = 500,
        legend = attr(orientation = "h", y = -0.22, x = 0.0),
        margin = attr(l = 70, r = 20, t = 60, b = 110),
    )

    return p.plot
end

t_highd = @elapsed highd_runtime_fig = highd_runtime_figure(profile_stats; dmin = 125)
println("high-d runtime figure build_seconds = ", round(t_highd, digits = 3),
        ", traces = ", length(highd_runtime_fig.data))

highd_runtime_html = joinpath(@__DIR__, "stratify-highd-runtime-report.html")
savefig(highd_runtime_fig, highd_runtime_html)
println("wrote ", highd_runtime_html)

t_pair = @elapsed highd_quickder_fig = highd_quickder_cpu_gpu_figure(profile_stats; dmin = 125)
println("high-d CPU/GPU figure build_seconds = ", round(t_pair, digits = 3),
        ", traces = ", length(highd_quickder_fig.data))

highd_quickder_html = joinpath(@__DIR__, "stratify-highd-quickder-cpu-gpu.html")
savefig(highd_quickder_fig, highd_quickder_html)
println("wrote ", highd_quickder_html)

t_highd_complex = @elapsed highd_complexity_fig = highd_complexity_figure(profile_stats; dmin = 125)
println("high-d complexity figure build_seconds = ", round(t_highd_complex, digits = 3),
        ", traces = ", length(highd_complexity_fig.data))

highd_complexity_html = joinpath(@__DIR__, "stratify-highd-complexity-report.html")
savefig(highd_complexity_fig, highd_complexity_html)
println("wrote ", highd_complexity_html)

t_pair_complex = @elapsed highd_quickder_complexity_fig = highd_quickder_complexity_cpu_gpu_figure(profile_stats; dmin = 125)
println("high-d CPU/GPU complexity figure build_seconds = ", round(t_pair_complex, digits = 3),
        ", traces = ", length(highd_quickder_complexity_fig.data))

highd_quickder_complexity_html = joinpath(@__DIR__, "stratify-highd-quickder-complexity-cpu-gpu.html")
savefig(highd_quickder_complexity_fig, highd_quickder_complexity_html)
println("wrote ", highd_quickder_complexity_html)

highd_runtime_fig
highd_quickder_fig
highd_complexity_fig
highd_quickder_complexity_fig

In [ ]:
# Smoothed local complexity estimate after the runtime view.
function complexity_smooth_figure(data)
    base = data[isfinite.(data.median_t) .& (data.median_t .> 0), :]
    isempty(base) && error("No valid runtime rows to estimate complexity")

    rows = DataFrame(profile=String[], method=String[], d=Int[], slope=Float64[], ma_slope=Float64[])

    for g in groupby(base, [:profile, :method])
        sub = sort(g, :d)
        ds = Int.(collect(sub.d))
        ts = Float64.(collect(sub.median_t))

        valid = isfinite.(ts) .& (ts .> 0) .& (ds .> 1)
        if count(valid) < 2
            continue
        end

        dsv = ds[valid]
        tsv = ts[valid]
        local_slope = diff(log10.(tsv)) ./ diff(log10.(dsv))
        d_slope = dsv[2:end]

        tail = 5
        for i in eachindex(local_slope)
            lo = max(1, i - tail + 1)
            avg = mean(local_slope[lo:i])
            push!(rows, (String(sub.profile[1]), String(sub.method[1]), d_slope[i], local_slope[i], avg))
        end
    end

    isempty(rows) && error("Not enough points per (profile, method) to estimate a smoothed local slope")

    rows.method_rank = [get(method_rank, m, typemax(Int)) for m in rows.method]
    sort!(rows, [:profile, :method_rank, :d])

    p = plot(
        rows,
        x = :d,
        y = :ma_slope,
        color = :method,
        facet_col = :profile,
        kind = "scatter",
        mode = "lines+markers",
    )

    nprof = length(unique(rows.profile))
    for k in 1:nprof
        xa = Symbol("xaxis", k == 1 ? "" : string(k))
        ya = Symbol("yaxis", k == 1 ? "" : string(k))
        p.plot.layout[xa] = attr(title = "d")
        p.plot.layout[ya] = attr(
            title = "moving avg local exponent e (tail 5)",
            showgrid = true,
        )
    end

    relayout!(p;
        title_text = "Smoothed local complexity exponent by method (tail 5)",
        width = 1200,
        height = 500,
        legend = attr(orientation = "h", y = -0.18, x = 0.0),
        margin = attr(l = 70, r = 20, t = 60, b = 90),
    )

    p.plot
end

complexity_smooth_fig = complexity_smooth_figure(profile_stats)
complexity_smooth_fig

Log-log scale to identify complexity


In [ ]:
# Local complexity estimate on a log-log runtime model.
# For t ≈ c * d^e, the correct local exponent is the slope of log10(t) against log10(d):
#   e ≈ Δlog10(t) / Δlog10(d)
# We compute that local slope and then smooth it with a trailing window of 5
# to suppress local artifacts while keeping the overall complexity trend.
function complexity_figure(data)
    base = data[isfinite.(data.median_t) .& (data.median_t .> 0), :]
    isempty(base) && error("No valid runtime rows to estimate complexity")

    rows = DataFrame(profile=String[], method=String[], d=Int[], slope=Float64[], ma_slope=Float64[])

    for g in groupby(base, [:profile, :method])
        sub = sort(g, :d)
        ds = Int.(collect(sub.d))
        ts = Float64.(collect(sub.median_t))

        valid = isfinite.(ts) .& (ts .> 0) .& (ds .> 1)
        if count(valid) < 2
            continue
        end

        dsv = ds[valid]
        tsv = ts[valid]
        local_slope = diff(log10.(tsv)) ./ diff(log10.(dsv))
        d_slope = dsv[2:end]

        tail = 5
        for i in eachindex(local_slope)
            lo = max(1, i - tail + 1)
            avg = mean(local_slope[lo:i])
            push!(rows, (String(sub.profile[1]), String(sub.method[1]), d_slope[i], local_slope[i], avg))
        end
    end

    isempty(rows) && error("Not enough points per (profile, method) to estimate a local slope")

    rows.method_rank = [get(method_rank, m, typemax(Int)) for m in rows.method]
    sort!(rows, [:profile, :method_rank, :d])

    p = plot(
        rows,
        x = :d,
        y = :ma_slope,
        color = :method,
        facet_col = :profile,
        kind = "scatter",
        mode = "lines+markers",
    )

    nprof = length(unique(rows.profile))
    for k in 1:nprof
        xa = Symbol("xaxis", k == 1 ? "" : string(k))
        ya = Symbol("yaxis", k == 1 ? "" : string(k))
        p.plot.layout[xa] = attr(title = "d")
        p.plot.layout[ya] = attr(
            title = "moving avg local exponent e (tail 5)",
            showgrid = true,
        )
    end

    relayout!(p;
        title_text = "Smoothed local complexity exponent by method (tail 5)",
        width = 1200,
        height = 500,
        legend = attr(orientation = "h", y = -0.18, x = 0.0),
        margin = attr(l = 70, r = 20, t = 60, b = 90),
    )

    p.plot
end

t_build = @elapsed complexity_fig = complexity_figure(profile_stats)
println("complexity figure build_seconds = ", round(t_build, digits = 3),
        ", traces = ", length(complexity_fig.data))

complexity_html = joinpath(@__DIR__, "stratify-complexity-report.html")
savefig(complexity_fig, complexity_html)
println("wrote ", complexity_html)

complexity_fig

In [ ]:
function accuracy_figure(data)
    plot_data = data[isfinite.(data.median_residual) .& (data.median_residual .> 0), :]
    isempty(plot_data) && error("No valid residual rows to plot")

    plot_data.method_rank = [get(method_rank, m, typemax(Int)) for m in plot_data.method]
    sort!(plot_data, [:profile, :method_rank, :d])

    p = plot(
        plot_data,
        x = :d,
        y = :median_residual,
        color = :method,
        facet_col = :profile,
        kind = "scatter",
        mode = "lines+markers",
    )

    if SHOW_ERROR_BARS
        for tr in p.plot.data
            m = String(get(tr.fields, :name, ""))
            sub = filter(:method => ==(m), plot_data)
            if !isempty(sub)
                tr["error_y"] = attr(
                    type = "data",
                    symmetric = false,
                    array = sub.max_residual .- sub.median_residual,
                    arrayminus = sub.median_residual .- sub.min_residual,
                    visible = true,
                    thickness = 1.2,
                    width = 2,
                )
            end
        end
    end

    profiles = sort(unique(plot_data.profile))
    for (k, profile_name) in enumerate(profiles)
        sub = filter(:profile => ==(profile_name), plot_data)
        isempty(sub) && continue
        target = occursin("Float32", profile_name) ? 1e-8 : 1e-16
        add_trace!(p, scatter(
            x = [minimum(sub.d), maximum(sub.d)],
            y = [target, target],
            mode = "lines",
            showlegend = false,
            line = attr(color = "#666666", width = 1, dash = "dash"),
            hoverinfo = "skip",
            xaxis = k == 1 ? "x" : "x$(k)",
            yaxis = k == 1 ? "y" : "y$(k)",
        ))
    end

    nprof = length(profiles)
    for k in 1:nprof
        xa = Symbol("xaxis", k == 1 ? "" : string(k))
        ya = Symbol("yaxis", k == 1 ? "" : string(k))
        p.plot.layout[xa] = attr(title = "d")
        p.plot.layout[ya] = attr(title = "residual", type = "log")
    end

    relayout!(p;
        title_text = "Median residual by method and dimension (MacStudio data)",
        width = 1200,
        height = 500,
        legend = attr(orientation = "h", y = -0.18, x = 0.0),
        margin = attr(l = 70, r = 20, t = 60, b = 90),
    )

    p.plot
end

t_build = @elapsed accuracy_fig = accuracy_figure(profile_stats)
println("accuracy figure build_seconds = ", round(t_build, digits = 3),
        ", traces = ", length(accuracy_fig.data))

accuracy_html = joinpath(@__DIR__, "stratify-accuracy-report.html")
savefig(accuracy_fig, accuracy_html)
println("wrote ", accuracy_html)

accuracy_fig

## Design intent of this report

This report is aligned with the current MacStudio benchmark pipeline:

- reads `stratify-timing-MacStudio.csv` and any `stratify-b*.csv` shard files
- derives profiles from `(eltype, solve_tol)` rather than requiring a `profile` input column
- summarizes repeated trials per `(profile, method, d)` using median and min/max whiskers
- compares methods only on successful rows (`status == "ok"`)
- keeps target rules explicit at `1e-8` for Float32 and `1e-16` for Float64

In [ ]:
profile_meta